# 05. Ejecución en Paralelo Multi-Caso y Comparación Masiva

Demuestra el uso de `ParallelRunner` (`ProcessPoolExecutor` con contexto `spawn`) para ejecutar en
paralelo múltiples pares `(caso, arquitectura)`, distribuir hilos/GPUs y consolidar el tablero de resultados.

In [2]:
import sys
from pathlib import Path

# Localiza PINN_V4 automáticamente aunque se ejecute desde dentro de notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Raíz PINN_V4:", ROOT)

Raíz PINN_V4: C:\usr\ths_mia_fiis\tesis_MIA_lk\tesis_MIA_2026-2_pinn


In [3]:
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from frontend.state import WorkspaceStore

store = WorkspaceStore(ROOT / "frontend" / "workspace", ROOT / "cases")
out_root = store.runs_dir / "nb05_parallel_batch"

tasks = [
    RunTask(
        name="mms_constant__M1",
        problem_source=str(ROOT / "cases" / "benchmarks" / "mms_constant" / "problem.json"),
        architecture="M1",
        output_dir=str(out_root / "mms_constant__M1"),
        config_kwargs={"adam_steps": 120, "lbfgs_steps": 10, "width": 20, "depth": 3},
        overwrite=True,
    ),
    RunTask(
        name="mms_constant__CUSPR",
        problem_source=str(ROOT / "cases" / "benchmarks" / "mms_constant" / "problem.json"),
        architecture="CUSPR",
        output_dir=str(out_root / "mms_constant__CUSPR"),
        config_kwargs={"adam_steps": 120, "lbfgs_steps": 10, "width": 20, "depth": 3},
        overwrite=True,
    ),
]

runner = ParallelRunner(max_workers=2)
summary = runner.run(tasks)
for row in summary:
    print(row)

{'name': 'mms_constant__M1', 'status': 'completed', 'output_dir': 'C:\\usr\\ths_mia_fiis\\tesis_MIA_lk\\tesis_MIA_2026-2_pinn\\frontend\\workspace\\runs\\nb05_parallel_batch\\mms_constant__M1', 'architecture': 'M1', 'case': 'mms_constant', 'device': 'xpu', 'parameters': 922, 'elapsed_s': 470.6674248999916, 'evaluation': {'pde_rms': {'domain': 0.22604265809059143}, 'constitutive_rms': {}, 'interfaces': [], 'boundary': [{'side': 'bottom', 'region': 'domain', 'type': 'dirichlet', 'rms': 0.2730448544025421, 'unit': 'K'}, {'side': 'top', 'region': 'domain', 'type': 'dirichlet', 'rms': 0.2480928599834442, 'unit': 'K'}, {'side': 'left', 'region': 'domain', 'type': 'dirichlet', 'rms': 0.2364533692598343, 'unit': 'K'}, {'side': 'right', 'region': 'domain', 'type': 'dirichlet', 'rms': 0.2709048092365265, 'unit': 'K'}], 'balance': {'region_derived_pct': {'domain': 4.002133626243879}, 'region_constitutive_pct': {'domain': 4.002133626243879}, 'global_derived_pct': 4.002133626243879, 'global_constit

In [4]:
import pandas as pd

rows = []
for r in summary:
    ev = r.get("evaluation") or {}
    rows.append({
        "Arquitectura": r["architecture"],
        "Estado": r["status"],
        "Parámetros": r["parameters"],
        "Tiempo (s)": round(r["elapsed_s"], 1),
        "Tmax pred (°C)": ev.get("Tmax_pred_C"),
        "Tmax ref (°C)": ev.get("Tmax_ref_C"),
        "Error Tmax (K)": ev.get("error_Tmax_abs_K"),
        "Error elevación (%)": ev.get("error_rise_pct"),
        "RMSE (K)": ev.get("rmse_K"),
        "Dist. hotspot (m)": ev.get("hotspot_distance_m"),
        "Error ejecución": r.get("error"),
    })

comparison = pd.DataFrame(rows).set_index("Arquitectura").T
comparison.round(3)

Arquitectura,M1,CUSPR
Estado,completed,completed
Parámetros,922,1002
Tiempo (s),470.7,473.2
Tmax pred (°C),48.372097,48.733616
Tmax ref (°C),49.997854,49.997854
Error Tmax (K),1.625757,1.264238
Error elevación (%),5.419578,4.214429
RMSE (K),1.149054,1.224509
Dist. hotspot (m),0.0,0.0
Error ejecución,None,None
